# EPI NEXUA VENTURES

An investor-grade Google Colab demo for **Nexua Ventures**.

This notebook is designed to show the strongest current EPI product story in a live setting:

- a workflow is recorded as a portable `.epi` case file
- the active rulebook is embedded into the artifact
- EPI explains what violated policy
- a human review is added without rewriting the original evidence
- a forged copy later becomes visibly untrustworthy

## What to look for

1. **Signed evidence**: the artifact is not just a log file.
2. **Policy and control outcomes**: the artifact explains what failed.
3. **Human review**: the case file can carry accountable review.
4. **Trust break**: tampering is visible.

## Presentation note

Use the blue form buttons to run the demo.
The heavy implementation is intentionally tucked away so the notebook reads like a product walkthrough, not a raw engineering notebook.


In [ ]:
# @title 1. Install EPI Recorder { display-mode: "form" }
import subprocess
import sys

PACKAGE_VERSION = '2.8.9'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', f'epi-recorder=={PACKAGE_VERSION}'],
    check=True,
)

import epi_core
print('Installed epi-recorder', epi_core.__version__)


## Live Demo Flow

The notebook is intentionally short:

- initialize a signed workspace
- define the rulebook
- create a clean artifact and a faulty artifact
- attach human review to the faulty case
- show the real embedded `.epi` viewer inside Colab
- compare trusted evidence vs tampered evidence


In [ ]:
# @title 2. Initialize Signed Demo Workspace { display-mode: "form" }
import html
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from IPython.display import HTML, display

from epi_cli.keys import KeyManager, generate_default_keypair_if_missing
from epi_core.container import EPIContainer
from epi_core.review import ReviewRecord, add_review_to_artifact, make_review_entry
from epi_core.trust import create_verification_report, verify_embedded_manifest_signature
from epi_recorder import record
from epi_recorder.integrations import OpenAIAgentsRecorder

WORKSPACE = Path('/content/epi_nexua_ventures')
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)

ARTIFACTS = WORKSPACE / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

generate_default_keypair_if_missing(console_output=False)
key_manager = KeyManager()
private_key = key_manager.load_private_key('default')


def run_cli(*args: str) -> str:
    result = subprocess.run(
        [sys.executable, '-m', 'epi_cli.main', *args],
        capture_output=True,
        text=True,
    )
    text = (result.stdout or '') + ('\n' + result.stderr if result.stderr else '')
    return text.strip()


def load_artifact(epi_path: Path) -> dict:
    with zipfile.ZipFile(epi_path, 'r') as zf:
        names = set(zf.namelist())
        return {
            'manifest': json.loads(zf.read('manifest.json').decode('utf-8')),
            'steps': [
                json.loads(line)
                for line in zf.read('steps.jsonl').decode('utf-8').splitlines()
                if line.strip()
            ],
            'analysis': json.loads(zf.read('analysis.json').decode('utf-8')) if 'analysis.json' in names else None,
            'policy': json.loads(zf.read('policy.json').decode('utf-8')) if 'policy.json' in names else None,
            'policy_evaluation': json.loads(zf.read('policy_evaluation.json').decode('utf-8')) if 'policy_evaluation.json' in names else None,
            'review': json.loads(zf.read('review.json').decode('utf-8')) if 'review.json' in names else None,
            'files': sorted(names),
        }


def build_trust_report(epi_path: Path) -> dict:
    manifest = EPIContainer.read_manifest(epi_path)
    integrity_ok, mismatches = EPIContainer.verify_integrity(epi_path)
    signature_valid, signer_name, _ = verify_embedded_manifest_signature(manifest)
    return create_verification_report(
        integrity_ok=integrity_ok,
        signature_valid=signature_valid,
        signer_name=signer_name,
        mismatches=mismatches,
        manifest=manifest,
    )


def trust_label(report: dict) -> str:
    if not report['integrity_ok']:
        return 'Tampered'
    if report['signature_valid']:
        return 'Signed'
    return 'Unsigned'


def compact_json(value) -> str:
    try:
        return json.dumps(value, ensure_ascii=False)
    except Exception:
        return str(value)


def render_case_view(epi_path: Path, label: str) -> str:
    artifact = load_artifact(epi_path)
    report = build_trust_report(epi_path)
    status = trust_label(report)
    color = '#0f766e' if status == 'Signed' else ('#b45309' if status == 'Unsigned' else '#b91c1c')

    analysis = artifact['analysis'] or {}
    summary = analysis.get('summary') or {}
    primary = analysis.get('primary_fault') or {}
    why = primary.get('why_it_matters') or summary.get('headline') or 'No analysis available.'
    headline = summary.get('headline') or primary.get('plain_english') or 'No primary finding'

    review = artifact['review'] or {}
    review_entry = (review.get('reviews') or [{}])[0]
    review_outcome = review_entry.get('outcome', 'No review')
    review_notes = review_entry.get('notes', 'No reviewer notes')

    evaluation = artifact['policy_evaluation'] or {}
    controls = evaluation.get('results') or []
    controls_html = ''.join(
        f"<li style='margin:6px 0;'><strong>{html.escape(item.get('rule_id', '?'))}</strong> - {html.escape(item.get('status', 'unknown'))}</li>"
        for item in controls[:4]
    ) or "<li>No control evaluation embedded</li>"

    steps_html = ''.join(
        f"<div style='padding:8px 0;border-top:1px solid #e5e7eb;'><strong>#{idx}</strong> {html.escape(step.get('kind', 'unknown'))}<div style='font-size:12px;color:#6b7280;margin-top:4px;'>{html.escape(compact_json(step.get('content', {}))[:220])}</div></div>"
        for idx, step in enumerate(artifact['steps'][:6], start=1)
    )

    files_html = ''.join(
        f"<li style='margin:4px 0;font-family:ui-monospace,monospace;font-size:12px;'>{html.escape(name)}</li>"
        for name in artifact['files']
    )

    return f'''
    <div style="font-family:system-ui,-apple-system,sans-serif;background:white;border:1px solid #d1d5db;border-radius:18px;overflow:hidden;box-shadow:0 10px 30px rgba(0,0,0,0.06);">
      <div style="background:{color};color:white;padding:18px 20px;">
        <div style="font-size:12px;font-weight:700;letter-spacing:0.06em;text-transform:uppercase;opacity:0.95;">{html.escape(label)}</div>
        <div style="font-size:30px;font-weight:800;margin-top:6px;">{html.escape(status)}</div>
        <div style="margin-top:8px;font-size:14px;opacity:0.95;">{html.escape(report['trust_message'])}</div>
      </div>
      <div style="padding:18px 20px;display:grid;grid-template-columns:1.1fr 0.9fr;gap:18px;">
        <div>
          <div style="background:#f8fafc;border:1px solid #e5e7eb;border-radius:14px;padding:14px;margin-bottom:14px;">
            <div style="font-size:12px;font-weight:700;text-transform:uppercase;color:#6b7280;">Workflow Goal</div>
            <div style="font-size:20px;font-weight:800;margin-top:6px;color:#111827;">{html.escape(artifact['manifest'].get('goal', 'Execution evidence'))}</div>
          </div>
          <div style="background:#f8fafc;border:1px solid #e5e7eb;border-radius:14px;padding:14px;margin-bottom:14px;">
            <div style="font-size:12px;font-weight:700;text-transform:uppercase;color:#6b7280;">Primary Finding</div>
            <div style="font-size:18px;font-weight:800;margin-top:6px;color:#111827;">{html.escape(headline)}</div>
            <div style="margin-top:8px;color:#475569;">{html.escape(why)}</div>
          </div>
          <div style="background:#f8fafc;border:1px solid #e5e7eb;border-radius:14px;padding:14px;">
            <div style="font-size:12px;font-weight:700;text-transform:uppercase;color:#6b7280;">Timeline Snapshot</div>
            <div style="margin-top:8px;color:#475569;">{steps_html}</div>
          </div>
        </div>
        <div>
          <div style="background:#f8fafc;border:1px solid #e5e7eb;border-radius:14px;padding:14px;margin-bottom:14px;">
            <div style="font-size:12px;font-weight:700;text-transform:uppercase;color:#6b7280;">Trust Facts</div>
            <div style="margin-top:8px;">Integrity: <strong>{'Intact' if report['integrity_ok'] else 'Compromised'}</strong></div>
            <div style="margin-top:6px;">Signature: <strong>{'Valid' if report['signature_valid'] else ('Missing' if report['signature_valid'] is None else 'Invalid')}</strong></div>
            <div style="margin-top:6px;">Trust level: <strong>{html.escape(report['trust_level'])}</strong></div>
          </div>
          <div style="background:#f8fafc;border:1px solid #e5e7eb;border-radius:14px;padding:14px;margin-bottom:14px;">
            <div style="font-size:12px;font-weight:700;text-transform:uppercase;color:#6b7280;">Control Outcomes</div>
            <ul style="margin:10px 0 0 18px;padding:0;color:#334155;">{controls_html}</ul>
          </div>
          <div style="background:#f8fafc;border:1px solid #e5e7eb;border-radius:14px;padding:14px;margin-bottom:14px;">
            <div style="font-size:12px;font-weight:700;text-transform:uppercase;color:#6b7280;">Human Review</div>
            <div style="margin-top:8px;">Outcome: <strong>{html.escape(review_outcome)}</strong></div>
            <div style="margin-top:6px;color:#475569;">{html.escape(review_notes)}</div>
          </div>
          <div style="background:#f8fafc;border:1px solid #e5e7eb;border-radius:14px;padding:14px;">
            <div style="font-size:12px;font-weight:700;text-transform:uppercase;color:#6b7280;">Files Inside Artifact</div>
            <ul style="margin:10px 0 0 18px;padding:0;color:#334155;">{files_html}</ul>
          </div>
        </div>
      </div>
    </div>
    '''


def render_side_by_side(cases):
    cards = ''.join(
        f"<div style='flex:1 1 0;min-width:320px;'>{render_case_view(path, label)}</div>"
        for path, label in cases
    )
    return f"<div style='display:flex;flex-wrap:wrap;gap:18px;align-items:stretch;margin:18px 0;'>{cards}</div>"


def render_command_output(title: str, output: str) -> str:
    return f"<div style='background:#0f172a;color:#e2e8f0;border-radius:16px;padding:16px 18px;margin:14px 0;'><div style='font-size:12px;font-weight:700;letter-spacing:0.06em;text-transform:uppercase;color:#93c5fd;'>{html.escape(title)}</div><pre style='white-space:pre-wrap;margin-top:10px;font-family:ui-monospace,monospace;font-size:12px;line-height:1.5;'>{html.escape(output)}</pre></div>"


def render_real_epi_viewer(epi_path: Path, extract_dir_name: str, title: str, height: int = 980):
    viewer_dir = WORKSPACE / extract_dir_name
    if viewer_dir.exists():
        shutil.rmtree(viewer_dir)

    result = subprocess.run(
        [sys.executable, '-m', 'epi_cli.main', 'view', str(epi_path), '--extract', str(viewer_dir)],
        capture_output=True,
        text=True,
        check=True,
    )
    viewer_html = (viewer_dir / 'viewer.html').read_text(encoding='utf-8')
    iframe_srcdoc = html.escape(viewer_html, quote=True)

    header = f'''
    <div style="background:linear-gradient(135deg,#eff6ff,#f8fafc);border:1px solid #bfdbfe;border-radius:18px;padding:20px 22px;margin:14px 0;">
      <div style="font-size:12px;font-weight:700;letter-spacing:0.06em;text-transform:uppercase;color:#1d4ed8;">Real Embedded Viewer</div>
      <div style="font-size:24px;font-weight:800;color:#0f172a;margin-top:6px;">{html.escape(title)}</div>
      <div style="margin-top:10px;color:#334155;">This is the actual <code>viewer.html</code> regenerated from the <code>.epi</code> artifact and rendered inside an iframe so Colab will execute the viewer's bundled scripts correctly.</div>
    </div>
    '''

    iframe = f'<iframe srcdoc="{iframe_srcdoc}" style="width:100%;height:{height}px;border:1px solid #d1d5db;border-radius:18px;background:white;box-shadow:0 10px 30px rgba(0,0,0,0.06);"></iframe>'
    display(HTML(header + iframe))
    output = ((result.stdout or '').strip() + ('\n' + result.stderr.strip() if result.stderr else '')).strip()
    if output:
        display(HTML(render_command_output('Viewer Extraction Output', output)))


print('Workspace ready:', WORKSPACE)
print('Artifacts directory:', ARTIFACTS)
print('Default signing keypair ready')


In [ ]:
# @title 3. Define The Rulebook { display-mode: "form" }
policy = {
    'policy_format_version': '2.0',
    'policy_id': 'nexua-ventures-refund-demo-v2',
    'system_name': 'nexua-ventures-refund-agent-demo',
    'system_version': '2.8.9',
    'policy_version': '2026-03-24',
    'scope': {
        'environment': 'nexua-demo',
        'workflow': 'nexua_customer_refund',
        'team': 'nexua-ventures-demo',
    },
    'approval_policies': [
        {
            'approval_id': 'manager-refund-approval',
            'required_roles': ['manager'],
            'minimum_approvers': 1,
            'reason_required': True,
        }
    ],
    'rules': [
        {
            'id': 'R005',
            'name': 'Manager Approval Before Refund',
            'severity': 'critical',
            'description': 'Refund approval requires an approved manager response.',
            'type': 'approval_guard',
            'mode': 'require_approval',
            'applies_at': ['decision'],
            'approval_action': 'approve_refund',
            'approval_policy_ref': 'manager-refund-approval',
        },
        {
            'id': 'R006',
            'name': 'Only Approved Refund Tools',
            'severity': 'high',
            'description': 'Only allowlisted tools may be called in this workflow.',
            'type': 'tool_permission_guard',
            'mode': 'detect',
            'applies_at': ['tool_call'],
            'allowed_tools': ['lookup_order', 'verify_identity', 'issue_refund', 'send_email'],
        },
    ],
}

policy_path = WORKSPACE / 'epi_policy.json'
policy_path.write_text(json.dumps(policy, indent=2), encoding='utf-8')
validate_output = run_cli('policy', 'validate', str(policy_path))

summary_html = f'''
<div style="background:linear-gradient(135deg,#eff6ff,#f8fafc);border:1px solid #bfdbfe;border-radius:18px;padding:20px 22px;margin:14px 0;">
  <div style="font-size:12px;font-weight:700;letter-spacing:0.06em;text-transform:uppercase;color:#1d4ed8;">Policy Ready</div>
  <div style="font-size:24px;font-weight:800;color:#0f172a;margin-top:6px;">Nexua Ventures Refund Control Rulebook</div>
  <div style="margin-top:10px;color:#334155;">This rulebook requires manager approval before refund approval and restricts the workflow to approved tools.</div>
  <ul style="margin:14px 0 0 18px;color:#334155;">
    <li>R005 - manager approval before refund decision</li>
    <li>R006 - only approved tools may run</li>
  </ul>
</div>
'''

display(HTML(summary_html))
display(HTML(render_command_output('Policy Validation Output', validate_output)))


In [ ]:
# @title 4. Create Signed Evidence And Human Review { display-mode: "form" }
def make_good_run(path: Path):
    with record(path, workflow_name='EPI NEXUA VENTURES Demo', goal='Resolve a customer refund safely') as epi:
        with epi.agent_run(
            'nexua-refund-agent',
            user_input='Refund order ORD-1001',
            session_id='sess-good-001',
            task_id='refund-ORD-1001',
            attempt=1,
        ) as agent:
            agent.plan('Look up the order, verify identity, request approval, then decide.')
            agent.message('user', 'Refund order ORD-1001')
            agent.memory_read('refund_history', query='ORD-1001', result_count=1)
            agent.tool_call('lookup_order', {'order_id': 'ORD-1001'})
            agent.tool_result('lookup_order', {'status': 'paid', 'amount': 120, 'customer_id': 'CUST-42'})
            agent.tool_call('verify_identity', {'customer_id': 'CUST-42'})
            agent.tool_result('verify_identity', {'verified': True})
            agent.approval_request('approve_refund', reason='Manual approval required for refund action')
            agent.approval_response('approve_refund', approved=True, reviewer='manager', notes='Approved after identity verification')
            agent.decision('approve_refund', confidence=0.94)
            agent.tool_call('issue_refund', {'order_id': 'ORD-1001', 'amount': 120})
            agent.tool_result('issue_refund', {'success': True, 'refund_id': 'RF-1001'})


def make_bad_run(path: Path):
    with record(path, workflow_name='EPI NEXUA VENTURES Demo', goal='Resolve a customer refund safely') as epi:
        with epi.agent_run(
            'nexua-refund-agent',
            user_input='Refund order ORD-2002',
            session_id='sess-bad-001',
            task_id='refund-ORD-2002',
            attempt=1,
        ) as agent:
            agent.plan('Move quickly and use the override if needed.')
            agent.message('user', 'Refund order ORD-2002')
            agent.tool_call('lookup_order', {'order_id': 'ORD-2002'})
            agent.tool_result('lookup_order', {'status': 'paid', 'amount': 500, 'customer_id': 'CUST-77'})
            agent.tool_call('admin_override', {'order_id': 'ORD-2002'})
            agent.tool_result('admin_override', {'status': 'override used'})
            agent.decision('approve_refund', confidence=0.62)
            agent.tool_call('issue_refund', {'order_id': 'ORD-2002', 'amount': 500})
            agent.tool_result('issue_refund', {'success': True, 'refund_id': 'RF-2002'})


good_artifact = ARTIFACTS / 'nexua_refund_agent_ok.epi'
bad_artifact = ARTIFACTS / 'nexua_refund_agent_missing_approval.epi'
reviewed_artifact = ARTIFACTS / 'nexua_refund_agent_reviewed.epi'

make_good_run(good_artifact)
make_bad_run(bad_artifact)
shutil.copy2(bad_artifact, reviewed_artifact)

analysis = load_artifact(reviewed_artifact)['analysis'] or {}
fault = analysis.get('primary_fault') or (analysis.get('secondary_flags') or [{}])[0]
review = ReviewRecord(
    reviewed_by='nexua-reviewer@epilabs.org',
    reviews=[
        make_review_entry(
            fault,
            'confirmed_fault',
            'Manager approval is missing and an unapproved tool was used. This workflow should have been escalated.',
            'nexua-reviewer@epilabs.org',
        )
    ],
)
review.sign(private_key)
add_review_to_artifact(reviewed_artifact, review)

summary_html = f'''
<div style="background:linear-gradient(135deg,#f8fafc,#ecfeff);border:1px solid #cbd5e1;border-radius:18px;padding:20px 22px;margin:14px 0;">
  <div style="font-size:12px;font-weight:700;letter-spacing:0.06em;text-transform:uppercase;color:#0f766e;">Artifacts Created</div>
  <div style="font-size:24px;font-weight:800;color:#0f172a;margin-top:6px;">One healthy case and one reviewed failing case</div>
  <ul style="margin:14px 0 0 18px;color:#334155;">
    <li><strong>{good_artifact.name}</strong> - expected to remain signed and clean</li>
    <li><strong>{reviewed_artifact.name}</strong> - expected to remain signed but contain a confirmed policy fault</li>
  </ul>
</div>
'''

display(HTML(summary_html))


In [ ]:
# @title 5. Render The Investor Case View { display-mode: "form" }
good_verify = run_cli('verify', str(good_artifact))
reviewed_analyze = run_cli('analyze', str(reviewed_artifact))
reviewed_verify = run_cli('verify', str(reviewed_artifact))

display(HTML(render_side_by_side([
    (good_artifact, 'Healthy Workflow'),
    (reviewed_artifact, 'Policy Failure With Human Review'),
])))

display(HTML(render_command_output('Healthy Artifact Verification', good_verify)))
display(HTML(render_command_output('Reviewed Artifact Analysis', reviewed_analyze)))
display(HTML(render_command_output('Reviewed Artifact Verification', reviewed_verify)))


In [ ]:
# @title 6. Show The Real Embedded .EPI Viewer { display-mode: "form" }
render_real_epi_viewer(
    reviewed_artifact,
    extract_dir_name='reviewed_artifact_viewer',
    title='Actual Viewer Bundled Inside The Reviewed Artifact',
)


In [ ]:
# @title 7. Tamper With The Reviewed Artifact { display-mode: "form" }
tampered_artifact = ARTIFACTS / 'nexua_refund_agent_tampered.epi'
shutil.copy2(reviewed_artifact, tampered_artifact)

with zipfile.ZipFile(tampered_artifact, 'a') as zf:
    analysis = json.loads(zf.read('analysis.json').decode('utf-8'))
    analysis['summary']['headline'] = 'FORGED ANALYSIS - DO NOT TRUST'
    zf.writestr('analysis.json', json.dumps(analysis, indent=2))

tampered_verify = run_cli('verify', str(tampered_artifact))

display(HTML(render_side_by_side([
    (reviewed_artifact, 'Reviewed Trusted Artifact'),
    (tampered_artifact, 'Forged Copy'),
])))

display(HTML(render_command_output('Tampered Artifact Verification', tampered_verify)))


In [ ]:
# @title 8. Optional OpenAI Agents Bridge Demo { display-mode: "form" }
bridge_artifact = ARTIFACTS / 'nexua_openai_agents_bridge_demo.epi'
synthetic_events = [
    {'type': 'run_item_stream_event', 'item': {'type': 'message', 'role': 'assistant', 'content': 'I will reset the password safely.'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'tool_call', 'tool_name': 'lookup_user', 'arguments': {'email': 'customer@example.com'}, 'call_id': 'call-1'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'tool_result', 'tool_name': 'lookup_user', 'result': {'user_id': 'user-123'}, 'call_id': 'call-1'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'approval_request', 'action': 'reset_password', 'reason': 'High-risk account action'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'approval_response', 'action': 'reset_password', 'approved': True, 'reviewer': 'manager', 'notes': 'Approved'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'final_output', 'output': 'Password reset completed successfully.'}},
]

with record(bridge_artifact, workflow_name='EPI NEXUA VENTURES OpenAI Agents Bridge Demo') as epi:
    with OpenAIAgentsRecorder(epi, agent_name='nexua-support-agent', user_input='Reset customer password') as recorder:
        for event in synthetic_events:
            recorder.consume(event)

display(HTML(render_side_by_side([
    (bridge_artifact, 'OpenAI Agents-Style Event Bridge'),
])))


In [ ]:
# @title 9. Download The Artifacts { display-mode: "form" }
from google.colab import files

print('Available artifacts:')
for path in sorted(ARTIFACTS.glob('*.epi')):
    print(' -', path.name)

# Uncomment any line below to download a file from Colab:
# files.download(str(good_artifact))
# files.download(str(reviewed_artifact))
# files.download(str(tampered_artifact))
# files.download(str(bridge_artifact))


## Final Takeaway

For a trust-sensitive AI workflow, EPI gives one portable artifact that can answer three questions:

- **What happened?**
- **What violated policy?**
- **Can this evidence still be trusted?**

That is the product story this notebook is built to show.
